In [1]:
#_________________TASK 1_________________

In [2]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta
from pprint import pprint
import random
import duckdb

np.random.seed(42)

user_id = 500000
today = pd.Timestamp("2026-03-03")
signup_dates = today - pd.to_timedelta(np.random.randint(0, 3*365, size=user_id),unit='D')
city = ["Baku", "Moscow", "Berlin", "Warsaw", "Kyiv", "Tokyo", "New York", "Chicago", "Los Angeles", "Otto"]
active = [True, False]

ds = pd.DataFrame({
    "user_id": range(1, user_id + 1),
    "city": np.random.choice(city, size=user_id),
    "score": np.random.uniform(0, 101, size=user_id),
    "active": np.random.choice(active, size=user_id),
    "signup_date": signup_dates,
    "age": np.random.randint(18, 81, size=user_id),
    "sessions": np.random.randint(0, 501, size=user_id),
    "revenue": np.random.uniform(0, 1001, size=user_id)
})

ds_parquet = ds.to_parquet("ds_parquet.parquet", engine="pyarrow", index=False) #12 kB
parquet_file = pq.ParquetFile("ds_parquet.parquet")

print("Number of row groups:", parquet_file.num_row_groups)
print("Number of rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)
print("Schema:", parquet_file.schema)

row_group = parquet_file.metadata.row_group(0)

for i in range(row_group.num_columns):
    col = row_group.column(i)
    stats = col.statistics
    print(f"Column {i}: {col.path_in_schema}")
    print(f"  Physical type: {col.physical_type}")
    print(f"  Compressed size: {col.total_compressed_size} bytes")
    print(f"  Min: {stats.min}")
    print(f"  Max: {stats.max}")
    print(f"  Null count: {stats.null_count}")
    print("")

ds_csv = ds.to_csv("ds_data.csv") #40 kB

ds_csv_kb = 40
ds_parquet_kb = 12
print(f"Ratio : {ds_csv_kb} / {ds_parquet_kb}", " = ", ds_csv_kb/ds_parquet_kb)

Number of row groups: 1
Number of rows: 500000
Number of columns: 8
Schema: <pyarrow._parquet.ParquetSchema object at 0x0000024AC2F648C0>
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}

Column 0: user_id
  Physical type: INT64
  Compressed size: 2280081 bytes
  Min: 1
  Max: 500000
  Null count: 0

Column 1: city
  Physical type: BYTE_ARRAY
  Compressed size: 251173 bytes
  Min: Baku
  Max: Warsaw
  Null count: 0

Column 2: score
  Physical type: DOUBLE
  Compressed size: 4279334 bytes
  Min: 0.00023078434695045225
  Max: 100.99943189997451
  Null count: 0

Column 3: act

In [3]:
"""
Parquet stores extra information called metadata. It knows the column names, types, min and max values, and null counts
CSV does not have this information. Metadata helps programs read only the data they need and makes queries faster
"""

'\nParquet stores extra information called metadata. It knows the column names, types, min and max values, and null counts\nCSV does not have this information. Metadata helps programs read only the data they need and makes queries faster\n'

In [4]:
#_________________TASK 2_________________

In [5]:
%time ds_full = pq.read_table("ds_parquet.parquet").to_pandas() #Wall time: 44.9 ms
%time ds_two_cols = pq.read_table("ds_parquet.parquet", columns=["user_id", "city"]).to_pandas() #Wall time: 27.7 ms
%time df_csv = pd.read_csv("ds_data.csv")[["user_id", "revenue"]] #Wall time: 235 ms

"""
Parquet stores each column in its own chunk. This means we can read only the columns we need
CSV stores data row by row, so we must read all columns even if we only need a few
Column chunks in Parquet make queries faster and use less memory
"""

CPU times: total: 46.9 ms
Wall time: 48 ms
CPU times: total: 15.6 ms
Wall time: 26.5 ms
CPU times: total: 234 ms
Wall time: 246 ms


'\nParquet stores each column in its own chunk. This means we can read only the columns we need\nCSV stores data row by row, so we must read all columns even if we only need a few\nColumn chunks in Parquet make queries faster and use less memory\n'

In [6]:
#_________________TASK 3_________________

In [7]:
%time df_age = pq.read_table("ds_parquet.parquet",filters=[("age", ">", 50)]).to_pandas() #Wall time: 39.4 ms
%time df_full = pq.read_table("ds_parquet.parquet").to_pandas() #Wall time: 44.9 ms
%time df_filtered = df_full[df_full["age"] > 50] #Wall time: 6.82 ms

%time print(df_age.shape[0]) #Wall time: 15.7 μs
%time print(df_filtered.shape[0]) #Wall time: 11.9 μs

"""
Predicate pushdown means the system applies a filter while reading the file
Only the rows that match the condition are loaded into memory
This is faster than reading all rows and filtering later because it skips unnecessary data
Using pushdown reduces memory use and speeds up queries, especially for large datasets
"""

%time result = duckdb.sql("SELECT * FROM 'ds_parquet.parquet' WHERE age > 50").df() #Wall time: 42.9 ms

CPU times: total: 31.2 ms
Wall time: 33.7 ms
CPU times: total: 62.5 ms
Wall time: 35.7 ms
CPU times: total: 15.6 ms
Wall time: 9.5 ms
237765
CPU times: total: 0 ns
Wall time: 20.3 μs
237765
CPU times: total: 0 ns
Wall time: 17.2 μs
CPU times: total: 46.9 ms
Wall time: 46.2 ms


In [8]:
#_________________TASK 4_________________

In [9]:
%time pandas_q1 = ds.groupby("city").size().reset_index(name="count")

print("")

print(pandas_q1)

print("")

duckdb_q1 = duckdb.sql(
"""SELECT city, COUNT(*) AS count
   FROM 'ds_parquet.parquet'
   GROUP BY city
   ORDER BY city"""
).df()

%time duckdb_q1

print("")

print(duckdb_q1)

print("")

%time pandas_q2 = ds.groupby("city")["score"].mean().reset_index().sort_values("score", ascending=False)

print("")

print(pandas_q2)

print("")

duckdb_q2 = duckdb.sql(
"""SELECT city, AVG(score) AS avg_score
   FROM 'ds_parquet.parquet'
   GROUP BY city
   ORDER BY avg_score DESC"""
).df()

%time duckdb_q2

print("")

print(duckdb_q2)

print("")

def pct_high_score(group):
    active_users = group[group["active"] == True]
    if len(active_users) == 0:
        return 0
    return (active_users["score"] > 75).mean() * 100

%time pandas_q3 = ds[ds["active"] == True].groupby("city").agg(pct_high_score=("score", lambda x: (x > 75).mean() * 100)).reset_index()

print("")

print(pandas_q3)

duckdb_q3 = duckdb.sql(
"""SELECT city,
          100.0 * SUM(CASE WHEN active=TRUE AND score>75 THEN 1 ELSE 0 END) / SUM(CASE WHEN active=TRUE THEN 1 ELSE 0 END) AS pct_high_score
   FROM 'ds_parquet.parquet'
   GROUP BY city
   ORDER BY city"""
).df()

%time duckdb_q3

print("") 

print(duckdb_q3)

print("")

%time pandas_q4 = ds.assign(rank=ds.groupby("city")["score"].rank(method="first", ascending=False)).query("rank <= 10").sort_values(["city", "rank"])

print("") 

print(pandas_q4[["city","user_id","score","rank"]])

print("") 

duckdb_q4 = duckdb.sql(
"""SELECT city, user_id, score, RANK() OVER (PARTITION BY city ORDER BY score DESC) AS rank
   FROM 'ds_parquet.parquet'
   QUALIFY rank <= 10
   ORDER BY city, rank"""
).df()

%time duckdb_q4

print("")

print(duckdb_q4)

print("")

%time pandas_q5 = ds.sort_values(["city","user_id"]).assign(running_total=lambda x: x.groupby("city")["score"].cumsum())

print("")

print(pandas_q5[["city","user_id","score","running_total"]].head(20))

print("")

duckdb_q5 = duckdb.sql(
"""SELECT city, user_id, score,
          SUM(score) OVER (PARTITION BY city ORDER BY user_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
   FROM 'ds_parquet.parquet'
   ORDER BY city, user_id"""
).df()

%time duckdb_q5

print("")

print(duckdb_q5.head(20))

"""
Pandas was easier for simple queries like counting or averaging because we can use `groupby` and built-in functions
DuckDB was easier for complex queries like ranking or running totals with SQL window functions
DuckDB was generally faster, especially on large datasets, because it reads only the needed data
The speed difference was biggest for queries with ranking or running totals, where pandas had to process all rows in memory
"""

CPU times: total: 31.2 ms
Wall time: 23.5 ms

          city  count
0         Baku  49917
1       Berlin  49947
2      Chicago  49862
3         Kyiv  50273
4  Los Angeles  50193
5       Moscow  50070
6     New York  49998
7         Otto  49853
8        Tokyo  50002
9       Warsaw  49885

CPU times: total: 0 ns
Wall time: 2.62 μs

          city  count
0         Baku  49917
1       Berlin  49947
2      Chicago  49862
3         Kyiv  50273
4  Los Angeles  50193
5       Moscow  50070
6     New York  49998
7         Otto  49853
8        Tokyo  50002
9       Warsaw  49885

CPU times: total: 15.6 ms
Wall time: 22.3 ms

          city      score
4  Los Angeles  50.747334
2      Chicago  50.728045
0         Baku  50.608172
5       Moscow  50.583650
7         Otto  50.577879
3         Kyiv  50.559982
1       Berlin  50.534361
8        Tokyo  50.453126
9       Warsaw  50.304835
6     New York  50.211198

CPU times: total: 0 ns
Wall time: 2.86 μs

          city  avg_score
0  Los Angeles  50.7473

'\nPandas was easier for simple queries like counting or averaging because we can use `groupby` and built-in functions\nDuckDB was easier for complex queries like ranking or running totals with SQL window functions\nDuckDB was generally faster, especially on large datasets, because it reads only the needed data\nThe speed difference was biggest for queries with ranking or running totals, where pandas had to process all rows in memory\n'

In [10]:
#_________________TASK 5_________________

In [21]:
np.random.seed(42)

user_id = 500000
today = pd.Timestamp("2026-03-03")
signup_dates = today - pd.to_timedelta(np.random.randint(0, 3*365, size=user_id),unit='D')
city = ["Baku", "Moscow", "Berlin", "Warsaw", "Kyiv", "Tokyo", "New York", "Chicago", "Los Angeles", "Otto"]
active = [True, False]

ds_task5 = pd.DataFrame({
    "user_id": range(1, user_id + 1),
    "city": np.random.choice(city, size=user_id),
    "score": np.random.uniform(0, 101, size=user_id),
    "active": np.random.choice(active, size=user_id),
    "signup_date": signup_dates,
    "age": np.random.randint(18, 81, size=user_id),
    "sessions": np.random.randint(0, 501, size=user_id),
    "revenue": np.random.uniform(0, 1001, size=user_id)
})

arrow_table = pa.Table.from_pandas(ds_task5)
print(ds_task5.dtypes) #city: object

print(arrow_table.schema) #city: string

pq.write_table(arrow_table, "arrow_table.parquet")

new_arrow_table = pq.read_table("arrow_table.parquet")

print(new_arrow_table)
print(new_arrow_table.schema)

df_from_arrow = new_arrow_table.to_pandas()

print(ds_task5.head())

print(df_from_arrow.head())

print("Data match check:", ds_task5.equals(df_from_arrow))

df_arrow_backend = pd.read_parquet("arrow_table.parquet", dtype_backend="pyarrow")

print(df_arrow_backend.dtypes)

df_traditional = pd.read_parquet("arrow_table.parquet")
print(df_traditional.dtypes)

"""
Arrow is a columnar in-memory format that acts as a bridge between storage and analysis tools
Parquet stores data on disk in a columnar way, Arrow can read it efficiently into memory
Pandas can use Arrow to handle data faster and with less memory, while DuckDB can query Arrow Tables or Parquet directly
This connection lets analysts move data between storage, Python, and SQL without copying or converting it repeatedly, making large-scale analytics much faster
"""

user_id                 int64
city                   object
score                 float64
active                   bool
signup_date    datetime64[ns]
age                     int32
sessions                int32
revenue               float64
dtype: object
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1163
pyarrow.Table
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
----
user_id: [[1,2,3,4,5,...,131068,131069,131070,131071,131072],[131073,131074,131075,131076,131077,...,262140,262141,262142,262143,262144],[262145,262146,262147,262148,262149,...,393212,393213,393214,393215,393216],[393217,393218,393219,393220,393221,...,499996,499997,499998,499999,500000]]
city: [["New York","Tokyo","Kyiv","Moscow","New York",...,"Tokyo","Otto","Baku","Ba